In [2]:
import pandas as pd

customers = pd.read_csv(r"C:\Users\alsed\projects\ecommerce-sales-intelligence\data\raw\olist_customers_dataset.csv")
orders = pd.read_csv(r"C:\Users\alsed\projects\ecommerce-sales-intelligence\data\raw\olist_orders_dataset.csv")

print(customers.head())
print(customers.info())

print(orders.head())
print(orders.info())

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---

In [3]:
import pandas as pd

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

def inspect(df, name):
    print(f"\n{'='*50}")
    print(f"TABLE: {name}")
    print(f"{'='*50}")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())

In [4]:
inspect(customers, "customers")
inspect(orders, "orders")
inspect(order_items, "order_items")
inspect(products, "products")
inspect(payments, "payments")


TABLE: customers
Shape: (99441, 5)

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate rows: 0

TABLE: orders
Shape: (99441, 8)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_d

In [5]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

In [6]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [7]:
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders[["delivery_time_days"]].head()

,delivery_time_days
0,8.0
1,13.0
2,9.0
3,13.0
4,2.0


In [8]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_time_days               2965
dtype: int64

In [9]:
orders_clean = orders[orders["order_status"] == "delivered"].copy()

orders_clean["order_status"].value_counts()

order_status
delivered    96478
Name: count, dtype: int64

In [10]:
orders_clean.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
delivery_time_days                8
dtype: int64

In [11]:
orders_clean["order_year"] = orders_clean["order_purchase_timestamp"].dt.year
orders_clean["order_month"] = orders_clean["order_purchase_timestamp"].dt.month

orders_clean[["order_year", "order_month"]].head()

,order_year,order_month
0,2017,10
1,2018,7
2,2018,8
3,2017,11
4,2018,2


In [12]:
orders_clean = orders_clean[orders_clean["delivery_time_days"].notna()]

orders_clean["delivery_time_days"].isnull().sum()

np.int64(0)

In [13]:
order_items["item_total_value"] = order_items["price"] + order_items["freight_value"]

order_items[["price", "freight_value", "item_total_value"]].head()

,price,freight_value,item_total_value
0,58.90,13.29,72.19
1,239.90,19.93,259.83
2,199.00,17.87,216.87
3,12.99,12.79,25.78
4,199.90,18.14,218.04


In [14]:
df = orders_clean.merge(customers, on="customer_id", how="left")
print("After customers merge:", df.shape)


df = df.merge(order_items, on="order_id", how="left")
print("After order_items merge:", df.shape)

df = df.merge(products, on="product_id", how="left")
print("After products merge:", df.shape)

After customers merge: (96470, 15)
After order_items merge: (110189, 22)
After products merge: (110189, 30)


In [15]:
df.isnull().sum().sort_values(ascending=False)

df.to_csv("../data/cleaned/ecommerce_cleaned_dataset.csv", index=False)

In [16]:
print("\n===== KPI SUMMARY (Currency: BRL - R$) =====")

# Core KPIs
total_revenue = df["item_total_value"].sum()
total_orders = df["order_id"].nunique()
total_customers = df["customer_id"].nunique()

print(f"\nTotal Revenue: {total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")

# Monthly revenue
monthly_revenue = (
    df.groupby("order_month")["item_total_value"]
    .sum()
    .reset_index()
    .rename(columns={"order_month": "Month", "item_total_value": "Revenue"})
)

print("\nMonthly Revenue:")
print(monthly_revenue)

# Top cities
top_cities = (
    df.groupby("customer_city")["item_total_value"]
    .sum()
    .reset_index()
    .sort_values(by="item_total_value", ascending=False)
    .head(10)
    .rename(columns={"customer_city": "City", "item_total_value": "Revenue"})
)

print("\nTop 10 Cities by Revenue:")
print(top_cities)


===== KPI SUMMARY (Currency: BRL - R$) =====

Total Revenue: 15,418,394.83
Total Orders: 96,470
Total Customers: 96,470

Monthly Revenue:
    Month     Revenue
0       1  1205369.83
1       2  1237407.73
2       3  1534929.19
3       4  1523691.33
4       5  1695431.92
5       6  1501499.33
6       7  1593585.60
7       8  1631324.00
8       9   701220.95
9      10   797607.67
10     11  1153229.37
11     12   843097.91

Top 10 Cities by Revenue:
                City     Revenue
3563       sao paulo  2107960.17
3126  rio de janeiro  1111732.21
449   belo horizonte   405950.51
553         brasilia   345199.05
1135        curitiba   238459.72
2936    porto alegre   214611.84
700         campinas   209002.90
3218        salvador   207713.30
1518       guarulhos   157615.53
2440         niteroi   135447.96


In [17]:
import plotly.express as px

fig = px.line(
    monthly_revenue,
    x="Month",
    y="Revenue",
    title="Monthly Revenue Trend"
)

fig.show()

Revenue shows steady growth from Month 1 to Month 5, reaching a peak around Month 5. After a relatively stable period, there is a sharp decline in Month 9, followed by a partial recovery towards the end of the year. This pattern may indicate seasonality or external factors affecting demand during late summer.

In [18]:
top_5_cities = top_cities.head(5)

import plotly.express as px

fig = px.bar(
    top_5_cities,
    x="City",
    y="Revenue",
    title="Top 5 Cities by Revenue"
)

fig.show()

São Paulo is the dominant contributor to revenue, significantly outperforming other cities. This indicates that sales are highly concentrated in major urban areas, suggesting potential opportunities to expand market presence in smaller cities.

In [19]:
# Average Order Value (correct way)
average_order_value = df.groupby("order_id")["item_total_value"].sum().mean()

# Average delivery time
average_delivery_time = df["delivery_time_days"].mean()

# Orders per customer
orders_per_customer = df["order_id"].nunique() / df["customer_id"].nunique()

print(f"Average Order Value: {average_order_value:,.2f}")
print(f"Average Delivery Time: {average_delivery_time:.2f} days")
print(f"Orders per Customer: {orders_per_customer:.2f}")

Average Order Value: 159.83
Average Delivery Time: 12.01 days
Orders per Customer: 1.00


In [20]:
#Add Delivery Time Distribution
import plotly.express as px

fig = px.histogram(
    df,
    x="delivery_time_days",
    nbins=30,
    title="Delivery Time Distribution"
)

fig.show()

### 🚚 Delivery Time Insight

Most deliveries are completed within a consistent range, with an average of around 12 days. However, the presence of longer delivery times suggests potential inefficiencies or delays in certain cases, which could impact customer satisfaction.